# Proyecto Modelos y Simulación de Sistemas I

In [1]:
%pip install autogluon polars numpy

Note: you may need to restart the kernel to use updated packages.


In [26]:
import polars as pl
import pandas as pd
import opendatasets as od
from autogluon.tabular import TabularPredictor

In [3]:
link = "https://www.kaggle.com/competitions/udea-ai-4-eng-20251-pruebas-saber-pro-colombia"
od.download(link)

Skipping, found downloaded files in "./udea-ai-4-eng-20251-pruebas-saber-pro-colombia" (use force=True to force download)


In [4]:
csv_file = "udea-ai-4-eng-20251-pruebas-saber-pro-colombia/train.csv"
csv_file_test = "udea-ai-4-eng-20251-pruebas-saber-pro-colombia/test.csv"

### Cargar el csv

In [5]:
train_df = pl.scan_csv(csv_file)
test_df = pl.scan_csv(csv_file_test)

In [6]:
first_two_rows = train_df.head(2).collect()

display(first_two_rows)

ID,PERIODO,ESTU_PRGM_ACADEMICO,ESTU_PRGM_DEPARTAMENTO,ESTU_VALORMATRICULAUNIVERSIDAD,ESTU_HORASSEMANATRABAJA,FAMI_ESTRATOVIVIENDA,FAMI_TIENEINTERNET,FAMI_EDUCACIONPADRE,FAMI_TIENELAVADORA,FAMI_TIENEAUTOMOVIL,ESTU_PRIVADO_LIBERTAD,ESTU_PAGOMATRICULAPROPIO,FAMI_TIENECOMPUTADOR,FAMI_TIENEINTERNET.1,FAMI_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,coef_1,coef_2,coef_3,coef_4
i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64
904256,20212,"""ENFERMERIA""","""BOGOTÁ""","""Entre 5.5 millones y menos de …","""Menos de 10 horas""","""Estrato 3""","""Si""","""Técnica o tecnológica incomple…","""Si""","""Si""","""N""","""No""","""Si""","""Si""","""Postgrado""","""medio-alto""",0.322,0.208,0.31,0.267
645256,20212,"""DERECHO""","""ATLANTICO""","""Entre 2.5 millones y menos de …","""0""","""Estrato 3""","""No""","""Técnica o tecnológica completa""","""Si""","""No""","""N""","""No""","""Si""","""No""","""Técnica o tecnológica incomple…","""bajo""",0.311,0.215,0.292,0.264


| Columna                      | Tipo de Variable      | Acción de Codificación                                                                                                                                   |
|------------------------------|-----------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------|
| `RENDIMIENTO_GLOBAL`         | Ordinal (Target)      | Mapear a valores numéricos `0`–`3` según el orden (e.g., `"bajo": 0`, `"medio-bajo": 1`, `"medio-alto": 2`, `"alto": 3`)                                  |
| `FAMI_ESTRATOVIVIENDA`       | Ordinal               | Mapear `"Estrato 1"`…`"Estrato 6"` a valores numéricos `1`…`6`                                                                                           |
| `ESTU_HORASSEMANATRABAJA`    | Ordinal               | Mapear rangos de texto (e.g., `"0"`, `"Menos de 10 horas"`, etc.) a valores numéricos ordinales                                                          |
| `FAMI_TIENEINTERNET`         | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8`                                                                                                      |
| `FAMI_TIENELAVADORA`         | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8`                                                                                                      |
| `FAMI_TIENEAUTOMOVIL`        | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8`                                                                                                      |
| `ESTU_PRIVADO_LIBERTAD`      | Categórica Binaria    | Convertir a `0` (N) / `1` (S) y castear a `UInt8`                                                                                                        |
| `ESTU_PAGOMATRICULAPROPIO`   | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8`                                                                                                      |
| `FAMI_TIENECOMPUTADOR`       | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8`                                                                                                      |
| `FAMI_TIENEINTERNET.1`       | Categórica Binaria    | Convertir a `0` (No) / `1` (Si) y castear a `UInt8` —  **Revisar si es columna duplicada**                                                                  |
| `FAMI_EDUCACIONPADRE`        | Categórica Nominal    | Aplicar One-hot Encoding o Target Encoding                                                                                                               |
| `FAMI_EDUCACIONMADRE`        | Categórica Nominal    | Aplicar One-hot Encoding o Target Encoding                                                                                                               |
| `ESTU_PRGM_ACADEMICO`        | Categórica Nominal    | Aplicar One-hot Encoding o Target Encoding                                                                                                               |
| `ESTU_PRGM_DEPARTAMENTO`     | Categórica Nominal    | Aplicar One-hot Encoding o Target Encoding                                                                                                               |
| `coef_1` … `coef_4`          | Continua              | Mantener como `Float64`, pero normalizar hacia 1                                                                                                                                 |
| Otras columnas (`ID`, `PERIODO`, etc.) | Variables de Identificación | No hacer nada por ahora                                                                                         |


## Preprocesamiento Parametrizado

hemos notado que en la versión anterior, no era trivial vovlerlo a aplicar a test, por lo tanto, hemos cambiado un poco el enfoque

### Definiciones de constantes y mapeos

In [7]:
TEXT_COLS_TO_NORMALIZE = [
    "ESTU_PRGM_DEPARTAMENTO", "ESTU_VALORMATRICULAUNIVERSIDAD",
    "ESTU_HORASSEMANATRABAJA", "FAMI_ESTRATOVIVIENDA", "FAMI_TIENEINTERNET",
    "FAMI_EDUCACIONPADRE", "FAMI_TIENELAVADORA", "FAMI_TIENEAUTOMOVIL",
    "ESTU_PRIVADO_LIBERTAD", "ESTU_PAGOMATRICULAPROPIO", "FAMI_TIENECOMPUTADOR",
    "FAMI_EDUCACIONMADRE"
]
ORDINAL_COLS = {
    "RENDIMIENTO_GLOBAL": ["bajo", "medio-bajo", "medio-alto", "alto"],
    "FAMI_ESTRATOVIVIENDA": ["sin estrato", "estrato 1", "estrato 2", "estrato 3", "estrato 4", "estrato 5", "estrato 6"],
    "ESTU_HORASSEMANATRABAJA": ["0", "menos de 10 horas", "entre 11 y 20 horas", "entre 21 y 30 horas", "mas de 30 horas"],
}
BINARY_COLS = [
    "FAMI_TIENEINTERNET", "FAMI_TIENELAVADORA", "FAMI_TIENEAUTOMOVIL",
    "ESTU_PRIVADO_LIBERTAD", "ESTU_PAGOMATRICULAPROPIO", "FAMI_TIENECOMPUTADOR",
]
NOMINAL_COLS = [
    "FAMI_EDUCACIONPADRE", "FAMI_EDUCACIONMADRE", "ESTU_PRGM_DEPARTAMENTO",
    "AREA_PROGRAMA"
]
PROGRAM_CORRECTION_MAPPING = {
    "ADMINISTRACIN DE EMPRESAS": "ADMINISTRACION DE EMPRESAS", "ADMINISTRACIN DE NEGOCIOS INTERNACIONALES": "ADMINISTRACION DE NEGOCIOS INTERNACIONALES", "ADMINISTRACIN LOGSTICA": "ADMINISTRACION LOGISTICA", "ADMINISTRACIN PBLICA": "ADMINISTRACION PUBLICA", "ADMINSITRACION DE EMPRESAS": "ADMINISTRACION DE EMPRESAS", "ADMINISTRACION DE EMPRESAS TURISTICA": "ADMINISTRACION DE EMPRESAS TURISTICAS", "ADMINISTRACION DE MERCADEO Y LOGISTICA INTERNACIONALES": "ADMINISTRACION EN MERCADEO Y LOGISTICA INTERNACIONALES", "ADMINISTRACION DE NEGOCIOS INTERNACIONALES": "ADMINISTRacion EN NEGOCIOS INTERNACIONALES", "ADMINISTRACION DE SERVICIOS DE SALUD": "ADMINISTRACION EN SERVICIOS DE SALUD", "CIENCIA POLITICA": "CIENCIAS POLITICAS", "COMUNICACIN SOCIAL": "COMUNICACION SOCIAL", "COMUNICACIN SOCIAL Y PERIODISMO": "COMUNICACION SOCIAL Y PERIODISMO", "COMUNICACIN SOCIAL PERIODISMO": "COMUNICACION SOCIAL Y PERIODISMO", "COMUNICACION SOCIALY PERIODISMO":"COMUNICACION SOCIAL Y PERIODISMO", "COMUNICACIN VISUAL": "COMUNICACION VISUAL", "COMUNICACION": "COMUNICACIONES", "COMUNICACION AUDIOVISUAL Y MULTIMEDIAL": "COMUNICACION AUDIOVISUAL Y MULTIMEDIOS", "CONTADURIA PBLICA": "CONTADURIA PUBLICA", "DEPORTE Y ACTIVIDADA FISICA": "DEPORTE Y ACTIVIDAD FISICA", "DISENO CROSSMEDIA": "DISENO CROSSMEDIA", "DISEO CROSSMEDIA": "DISENO CROSSMEDIA", "DISENO DE MODA": "DISENO DE MODAS", "DISEÑO GRAFICO": "DISENO GRAFICO", "ECONOMA": "ECONOMIA", "INGENIERA DE SISTEMAS": "INGENIERIA DE SISTEMAS", "INGENIERA ELCTRICA": "INGENIERIA ELECTRICA", "INGENIERA EN SOFTWARE": "INGENIERIA EN SOFTWARE", "INGENIERA INDUSTRIAL": "INGENIERIA INDUSTRIAL", "INGENIERA INFORMTICA": "INGENIERIA INFORMATICA", "INGENIERIA DE CONTROL": "INGENIERIA EN CONTROL", "INGENIERIA DE PROCESOS INDUSTRIALES": "INGENIERIA EN PROCESOS INDUSTRIALES", "INGENIERIA DE SOFTWARE": "INGENIERIA EN SOFTWARE", "INGENIIERIA DE SOFTWARE": "INGENIERIA DE SOFTWARE", "INGENIERIA DE TELECOMUNICACIONES": "INGENIERIA EN TELECOMUNICACIONES", "INGENIERIA EN ENERGIA": "INGENIERIA EN ENERGIAS", "INGENIERIA MECATRONICO": "INGENIERIA MECATRONICA", "INTRUMENTACION QUIRURGICA": "INSTRUMENTACION QUIRURGICA", "LICENCIATURA EN ARTES ESCNICAS": "LICENCIATURA EN ARTES ESCENICAS", "LICENCIATURA EN EDUCACIN ARTSTICA": "LICENCIATURA EN EDUCACION ARTISTICA", "LICENCIATURA EN EDUCACIN BSICA PRIMARIA": "LICENCIATURA EN EDUCACION BASICA PRIMARIA", "LICENCIATURA EN EDUCACIN INFANTIL": "LICENCIATURA EN EDUCACION INFANTIL", "LICENCIATURA EN EDUCACION FISICA RECREACION Y DEPORTE": "LICENCIATURA EN EDUCACION FISICA RECREACION Y DEPORTES", "LICENCIATURA EN EDUCACION FISICARECREACION Y DEPORTE": "LICENCIATURA EN EDUCACION FISICA RECREACION Y DEPORTES", "LICENCIATURA EN EDUCACON FISICA RECREACION Y DEPORTES": "LICENCIATURA EN EDUCACION FISICA RECREACION Y DEPORTES", "LICENCIATURA EN FILOSOFA Y HUMANIDADES": "LICENCIATURA EN FILOSOFIA Y HUMANIDADES", "LICENCIATURA EN LENGUAS EXTRANJERAS CON NFASIS EN INGLS": "LICENCIATURA EN LENGUAS EXTRANJERAS CON ENFASIS EN INGLES", "LICENCIATURA EN LENGUAS EXTRANJERAS INGLESFRANCES": "LICENCIATURA EN LENGUAS EXTRANJERAS INGLES FRANCES", "LICENCIATURA EN MATEMATICA APLICADA": "LICENCIATURA EN MATEMATICAS APLICADAS", "LICENCIATURA EN MATEMTICAS": "LICENCIATURA EN MATEMATICAS APLICADAS", "LICENCIATURA EN PEDAGOGA INFANTIL": "LICENCIATURA EN PEDAGOGIA INFANTIL", "CIENCIA DE LA INFORMACION BIBLIOTECOLOGIA": "CIENCIA DE LA INFORMACION Y BIBLIOTECOLOGIA", "GEOLOGA": "GEOLOGIA", "PROFESIONAL EN GASTRONOMA": "PROFESIONAL EN GASTRONOMIA", "PSICOLOGA": "PSICOLOGIA", "QUMICA FARMACUTICA": "QUIMICA FARMACEUTICA"
}
PROGRAM_AREA_MAPPING = {
    "ADMINISTRACION DE EMPRESAS": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION DE NEGOCIOS INTERNACIONALES": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION EN NEGOCIOS INTERNACIONALES": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION LOGISTICA": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION PUBLICA": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION DE EMPRESAS TURISTICAS": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION EN MERCADEO Y LOGISTICA INTERNACIONALES": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ADMINISTRACION TURISTICA Y HOTELERA": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "CONTADURIA PUBLICA": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "ECONOMIA": "ECONOMIA, ADMINISTRACION Y CONTADURIA", "INGENIERIA DE SISTEMAS": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA EN SOFTWARE": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA INFORMATICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA EN TELECOMUNICACIONES": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA ELECTRONICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA ELECTRICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA MECATRONICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA EN CONTROL": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA INDUSTRIAL": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA EN PROCESOS INDUSTRIALES": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA BIOLOGICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA GEOLOGICA": "INGENIERIA, ARQUITECTURA Y URBANISMO", "INGENIERIA EN ENERGIAS": "INGENIERIA, ARQUITECTURA Y URBANISMO", "ADMINISTRACION EN SERVICIOS DE SALUD": "CIENCIAS DE LA SALUD", "INSTRUMENTACION QUIRURGICA": "CIENCIAS DE LA SALUD", "QUIMICA FARMACEUTICA": "CIENCIAS DE la SALUD", "CIENCIAS POLITICAS": "CIENCIAS SOCIALES Y HUMANIDADES", "COMUNICACION SOCIAL": "CIENCIAS SOCIALES Y HUMANIDADES", "COMUNICACION SOCIAL Y PERIODISMO": "CIENCIAS SOCIALES Y HUMANIDADES", "COMUNICACIONES": "CIENCIAS SOCIALES Y HUMANIDADES", "COMUNICACION AUDIOVISUAL Y MULTIMEDIOS": "CIENCIAS SOCIALES Y HUMANIDADES", "COMUNICACION VISUAL": "CIENCIAS SOCIALES Y HUMANIDADES", "CIENCIA DE LA INFORMACION Y BIBLIOTECOLOGIA": "CIENCIAS SOCIALES Y HUMANIDADES", "PSICOLOGIA": "CIENCIAS SOCIALES Y HUMANIDADES", "TEOLOGIA": "CIENCIAS SOCIALES Y HUMANIDADES", "LICENCIATURA EN ARTES ESCENICAS": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN BIOLOGIA": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN EDUCACION ARTISTICA": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN EDUCACION BASICA PRIMARIA": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN EDUCACION INFANTIL": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN PEDAGOGIA INFANTIL": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN EDUCACION BASICA CON ENFASIS EN EDUCACION FISICA RECREACION Y DEPORTES": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN EDUCACION FISICA RECREACION Y DEPORTES": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN FILOSOFIA Y HUMANIDADES": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN INGLES ESPANOL": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN LENGUAS EXTRANJERAS CON ENFASIS EN INGLES": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN LENGUAS EXTRANJERAS INGLES FRANCES": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN MATEMATICAS APLICADAS": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN FISICA": "CIENCIAS DE LA EDUCACION", "LICENCIATURA EN MUSICA": "CIENCIAS DE LA EDUCACION", "DISENO CROSSMEDIA": "ARTES Y DISENO", "DISENO DE MODAS": "ARTES Y DISENO", "DISENO GRAFICO": "ARTES Y DISENO", "BIOLOGIA": "CIENCIAS BASICAS Y NATURALES", "ECOLOGIA": "CIENCIAS BASICAS Y NATURALES", "GEOLOGIA": "CIENCIAS BASICAS Y NATURALES", "ASTRONOMIA": "CIENCIAS BASICAS Y NATURALES", "AGRONOMIA": "AGRONOMIA, VETERINARIA Y AFINES", "PROFESIONAL EN GASTRONOMIA": "AGRONOMIA, VETERINARIA Y AFINES", "DEPORTE Y ACTIVIDAD FISICA": "DEPORTE Y EDUCACION FISICA"
}
ACCENT_MAP_GENERAL = {
    r"[áÁ]": "a", r"[éÉ]": "e", r"[íÍ]": "i", r"[óÓ]": "o",
    r"[úÚ]": "u", r"[üÜ]": "u", r"[ñÑ]": "n"
}

### Funciones de transformación

In [8]:
def limpiar_y_agrupar_programas(df: pl.LazyFrame) -> pl.LazyFrame:
    accent_map_programas = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "ñ": "n", "Ñ": "N"}
    if 'ESTU_PRGM_ACADEMICO' not in df.collect_schema().names(): return df
    prog_expr = pl.col("ESTU_PRGM_ACADEMICO").fill_null("SIN INFORMACION").str.to_uppercase()
    for o, r in accent_map_programas.items(): prog_expr = prog_expr.str.replace_all(o, r, literal=True)
    prog_expr = (prog_expr.str.replace_all(r'[^A-Z0-9 ]', '', literal=False).str.replace_all(r'\s{2,}', ' ', literal=False).str.strip_chars()
                 .replace_strict(PROGRAM_CORRECTION_MAPPING, default=pl.col("ESTU_PRGM_ACADEMICO")).alias("prog_norm"))
    df_with_norm = df.with_columns(prog_expr)
    area_expr = (pl.col("prog_norm").replace_strict(PROGRAM_AREA_MAPPING, default=pl.lit("OTRAS AREAS")).alias("AREA_PROGRAMA"))
    return df_with_norm.with_columns(area_expr)


In [9]:
def normalizar_columnas_texto(df: pl.LazyFrame) -> pl.LazyFrame:
    exprs = []
    available_cols = df.collect_schema().names()
    for col_name in TEXT_COLS_TO_NORMALIZE:
        if col_name in available_cols:
            expr = pl.col(col_name).fill_null("sin informacion").str.strip_chars().str.to_lowercase()
            for pat, rep in ACCENT_MAP_GENERAL.items(): expr = expr.str.replace_all(pat, rep)
            if col_name == "ESTU_PRIVADO_LIBERTAD": expr = expr.str.replace("n", "no").str.replace("s", "si").str.replace("í", "i")
            exprs.append(expr.alias(col_name))
    return df.with_columns(exprs)

In [10]:
def codificar_variables_binarias(df: pl.LazyFrame) -> pl.LazyFrame:
    binary_lookup = pl.DataFrame({"value": ["no", "si"], "bin": [0, 1]}).lazy()
    df_mapped = df
    available_cols = df.collect_schema().names()
    for col in BINARY_COLS:
        if col in available_cols:
            col_lower = col.lower()
            df_mapped = (df_mapped.rename({col: "value"}).join(binary_lookup, on="value", how="left")
                         .rename({"bin": f"{col_lower}_bin", "value": col}))
    return df_mapped

In [11]:
def codificar_variables_ordinales(df: pl.LazyFrame) -> pl.LazyFrame:
    df_mapped = df
    available_cols = df.collect_schema().names()
    for col, order in ORDINAL_COLS.items():
        if col in available_cols:
            lookup_df = pl.DataFrame({col: order}).with_row_index(f"{col.lower()}_ord").lazy()
            df_mapped = df_mapped.join(lookup_df, on=col, how="left")
    return df_mapped

In [12]:
def imputar_valores_faltantes(df: pl.DataFrame) -> pl.DataFrame:
    cols_con_nulos = ["fami_estratovivienda_ord", "estu_horassemanatrabaja_ord", "fami_tieneinternet_bin",
                      "fami_tienecomputador_bin", "fami_tienelavadora_bin", "estu_pagomatriculapropio_bin",
                      "fami_tieneautomovil_bin"]
    fill_exprs = []
    for col_name in cols_con_nulos:
        if col_name in df.columns:
            moda = df.get_column(col_name).mode().item()
            fill_exprs.append(pl.col(col_name).fill_null(moda))
    return df.with_columns(fill_exprs)

### Pipeline de transformación

In [13]:
def aplicar_transformaciones_base(df: pl.DataFrame) -> pl.DataFrame:
    """Orquesta el pipeline de limpieza y preprocesamiento base."""
    df_lazy = df.lazy()
    if "FAMI_TIENEINTERNET.1" in df_lazy.collect_schema().names():
        df_lazy = df_lazy.drop("FAMI_TIENEINTERNET.1")
        
    df_transformed = (df_lazy.pipe(limpiar_y_agrupar_programas).pipe(normalizar_columnas_texto)
                      .pipe(codificar_variables_ordinales).pipe(codificar_variables_binarias).collect())
    
    df_imputado = imputar_valores_faltantes(df_transformed)
    return df_imputado

## Aplicando Transformaciones

### Train y Test

In [14]:
print("Aplicando limpieza y preprocesamiento base a los datos de entrenamiento...")
train_df_limpio = aplicar_transformaciones_base(train_df)
print("Aplicando limpieza y preprocesamiento base a los datos de prueba...")
test_df_limpio = aplicar_transformaciones_base(test_df)

Aplicando limpieza y preprocesamiento base a los datos de entrenamiento...
Aplicando limpieza y preprocesamiento base a los datos de prueba...


In [28]:
display(train_df_limpio.head(15))

ID,PERIODO,ESTU_PRGM_ACADEMICO,ESTU_PRGM_DEPARTAMENTO,ESTU_VALORMATRICULAUNIVERSIDAD,ESTU_HORASSEMANATRABAJA,FAMI_ESTRATOVIVIENDA,FAMI_TIENEINTERNET,FAMI_EDUCACIONPADRE,FAMI_TIENELAVADORA,FAMI_TIENEAUTOMOVIL,ESTU_PRIVADO_LIBERTAD,ESTU_PAGOMATRICULAPROPIO,FAMI_TIENECOMPUTADOR,FAMI_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,coef_1,coef_2,coef_3,coef_4,prog_norm,AREA_PROGRAMA,rendimiento_global_ord,fami_estratovivienda_ord,estu_horassemanatrabaja_ord,fami_tieneinternet_bin,fami_tienelavadora_bin,fami_tieneautomovil_bin,estu_privado_libertad_bin,estu_pagomatriculapropio_bin,fami_tienecomputador_bin
i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,str,str,u32,u32,u32,i64,i64,i64,i64,i64,i64
904256,20212,"""ENFERMERIA""","""bogota""","""entre 5.5 millones y menos de …","""menos de 10 horas""","""estrato 3""","""si""","""tecnica o tecnologica incomple…","""si""","""si""","""no""","""no""","""si""","""postgrado""","""medio-alto""",0.322,0.208,0.31,0.267,"""ENFERMERIA""","""OTRAS AREAS""",2,3,1,1,1,1,0,0,1
645256,20212,"""DERECHO""","""atlantico""","""entre 2.5 millones y menos de …","""0""","""estrato 3""","""no""","""tecnica o tecnologica completa""","""si""","""no""","""no""","""no""","""si""","""tecnica o tecnologica incomple…","""bajo""",0.311,0.215,0.292,0.264,"""DERECHO""","""OTRAS AREAS""",0,3,0,0,1,0,0,0,1
308367,20203,"""MERCADEO Y PUBLICIDAD""","""bogota""","""entre 2.5 millones y menos de …","""mas de 30 horas""","""estrato 3""","""si""","""secundaria (bachillerato) comp…","""si""","""no""","""no""","""no""","""no""","""secundaria (bachillerato) comp…","""bajo""",0.297,0.214,0.305,0.264,"""MERCADEO Y PUBLICIDAD""","""OTRAS AREAS""",0,3,4,1,1,0,0,0,0
470353,20195,"""ADMINISTRACION DE EMPRESAS""","""santander""","""entre 4 millones y menos de 5.…","""0""","""estrato 4""","""si""","""no sabe""","""si""","""no""","""no""","""no""","""si""","""secundaria (bachillerato) comp…","""alto""",0.485,0.172,0.252,0.19,"""ADMINISTRACION DE EMPRESAS""","""ECONOMIA, ADMINISTRACION Y CON…",3,4,0,1,1,0,0,0,1
989032,20212,"""PSICOLOGIA""","""antioquia""","""entre 2.5 millones y menos de …","""entre 21 y 30 horas""","""estrato 3""","""si""","""primaria completa""","""si""","""si""","""no""","""no""","""si""","""primaria completa""","""medio-bajo""",0.316,0.232,0.285,0.294,"""PSICOLOGIA""","""CIENCIAS SOCIALES Y HUMANIDADE…",1,3,3,1,1,1,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
273010,20203,"""PSICOLOGIA""","""sucre""","""entre 1 millon y menos de 2.5 …","""entre 11 y 20 horas""","""estrato 1""","""si""","""tecnica o tecnologica completa""","""si""","""no""","""no""","""no""","""si""","""secundaria (bachillerato) inco…","""bajo""",0.151,0.399,0.241,0.292,"""PSICOLOGIA""","""CIENCIAS SOCIALES Y HUMANIDADE…",0,1,2,1,1,0,0,0,1
738026,20203,"""ADMINISTRACIÓN FINANCIERA""","""bogota""","""entre 500 mil y menos de 1 mil…","""mas de 30 horas""","""estrato 2""","""si""","""educacion profesional incomple…","""si""","""no""","""no""","""no""","""si""","""postgrado""","""medio-bajo""",0.212,0.284,0.283,0.324,"""ADMINISTRACIÓN FINANCIERA""","""OTRAS AREAS""",1,2,4,1,1,0,0,0,1
858669,20183,"""HOTELERIA Y TURISMO""","""bogota""","""entre 2.5 millones y menos de …","""mas de 30 horas""","""estrato 3""","""si""","""secundaria (bachillerato) comp…","""si""","""si""","""no""","""no""","""si""","""primaria completa""","""medio-bajo""",0.285,0.249,0.278,0.289,"""HOTELERIA Y TURISMO""","""OTRAS AREAS""",1,3,4,1,1,1,0,0,1


In [ ]:
final_feature_cols = [
    # Columnas categóricas originales (AutoGluon las maneja automáticamente)
    'FAMI_ESTRATOVIVIENDA', 'ESTU_HORASSEMANATRABAJA', 'FAMI_TIENEINTERNET',
    'FAMI_EDUCACIONPADRE', 'FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL',
    'ESTU_PRIVADO_LIBERTAD', 'ESTU_PAGOMATRICULAPROPIO', 'FAMI_TIENECOMPUTADOR',
    'FAMI_EDUCACIONMADRE', 'ESTU_PRGM_DEPARTAMENTO',
    'prog_norm', 'PERIODO'
    # Los coeficientes numéricos
    'coef_1', 'coef_2', 'coef_3', 'coef_4'
]

In [16]:
target_col = 'RENDIMIENTO_GLOBAL'

Dataframes para AutoGluon

In [17]:
train_data_ag = train_df_limpio.select(final_feature_cols + [target_col])
test_data_ag = test_df_limpio.select(final_feature_cols + ['ID']) 

In [18]:
train_data_pd = train_data_ag.to_pandas()
test_data_pd = test_data_ag.to_pandas()
test_data_pd_indexed = test_data_pd.set_index('ID')

### Ejecución AutoGluon

In [ ]:
save_path = 'autogluon_final_models_2'

Nuestro objetivo es maximizar el accuracy, por lo que usamos 'accuracy' como métrica de evaluación.

In [20]:
predictor = TabularPredictor(
    label=target_col,
    path=save_path,
    eval_metric='accuracy'  
)

In [ ]:
time_limit_seconds = 2700 # 30 minutos.

print(f"Iniciando entrenamiento de AutoML por {time_limit_seconds} segundos...")

Iniciando entrenamiento de AutoML por 60 segundos...


In [22]:
predictor.fit(
    train_data=train_data_pd,
    time_limit=time_limit_seconds,
    presets='best_quality',  # La máxima calidad posible
    ag_args_fit={'num_gpus': 1} 
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Tue Nov 5 00:21:55 UTC 2024
CPU Count:          16
Memory Avail:       7.71 GB / 15.15 GB (50.9%)
Disk Space Avail:   911.37 GB / 1006.85 GB (90.5%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked overfitting.

In [23]:
print("\nTabla de clasificación de modelos (Leaderboard):")
leaderboard = predictor.leaderboard(train_data_pd, silent=True)
display(leaderboard)


Tabla de clasificación de modelos (Leaderboard):


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,KNeighborsDist_BAG_L1,0.990455,0.282001,accuracy,1.839981,2.120154,0.684175,1.839981,2.120154,0.684175,1,True,2
1,KNeighborsUnif_BAG_L1,0.513656,0.286822,accuracy,1.823551,2.137669,0.680742,1.823551,2.137669,0.680742,1,True,1
2,LightGBMXT_BAG_L1,0.377460,0.375279,accuracy,0.874128,0.642153,17.200603,0.874128,0.642153,17.200603,1,True,3
3,WeightedEnsemble_L2,0.377460,0.375279,accuracy,0.885171,0.664906,19.432954,0.011043,0.022753,2.232351,2,True,4
4,WeightedEnsemble_L3,0.377353,0.375646,accuracy,4.936001,5.077228,35.157084,0.021431,0.022531,2.820281,3,True,6
5,LightGBMXT_BAG_L2,0.371006,0.368339,accuracy,4.914570,5.054697,32.336804,0.376909,0.154720,13.771284,2,True,5


Predicciones usando el mejor modelo

In [24]:
final_predictions = predictor.predict(test_data_pd_indexed)
print("   - Predicciones generadas.")

   - Predicciones generadas.


## Persistencia

In [27]:
ids_finales = final_predictions.index
predicciones_etiquetas = final_predictions.values
submission_df_pd = pd.DataFrame({"ID": ids_finales, "RENDIMIENTO_GLOBAL": predicciones_etiquetas})
submission_df_pd.to_csv("submission_autogluon_final.csv", index=False)

print("\n✅ ¡Proceso finalizado! 'submission_autogluon_final.csv' está listo.")
display(submission_df_pd.head())


✅ ¡Proceso finalizado! 'submission_autogluon_final.csv' está listo.


,ID,RENDIMIENTO_GLOBAL
0,550236,bajo
1,98545,alto
2,499179,alto
3,782980,bajo
4,785185,medio-bajo
